In [3]:
### What we want: get indexes from each industry and use their returns for fitting and showing a copula for the normal and t distributions

# This will eliminate survivorship bias, as these indices will be around for a while and don't die during market crashes

# will use training data from the mid 2000s to create a copula before the 2008 crash. Will do the same for 2020 and 2001
    # will then compare these results and what the normal and t copulas dictate in these scenarios

In [4]:
import yfinance as yf
import sys
import pandas as pd
import numpy as np
from scipy.stats import t, multivariate_t, norm
sys.path.append("/Users/willneuner/Desktop/FINTECH545") 

In [5]:
yahoo_tickers = {
    # Sectors / Industries
    "Technology": "XLK",
    "Financials": "XLF",
    "Energy": "XLE",
    "Healthcare": "XLV",
    "Industrials": "XLI",
    "Utilities": "XLU",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Materials": "XLB",
    "Real Estate": "XLRE",
    "Communication Services": "XLC",

    # Commodities
    "Broad Commodities": "DBC",
    "Crude Oil": "USO",
    "Gold": "GLD",
    # "Silver": "SLV",
    "Natural Gas": "UNG",
    "Agriculture": "DBA",

    # FX / Currencies
    "US Dollar Index": "UUP",
    "EUR/USD": "FXE",
    # "GBP/USD": "FXB",
    "JPY/USD": "FXY",
    # "CAD/USD": "FXC"
}

In [6]:
ticker = yahoo_tickers['Technology']
tickers = list(yahoo_tickers.values())
data = yf.download(tickers = tickers, period="5y", interval="1d")

/var/folders/1d/dkqlxg7d3yv8ccfw3vkrtnsm0000gn/T/ipykernel_22815/2551565624.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers = tickers, period="5y", interval="1d")
[*********************100%***********************]  19 of 19 completed


In [7]:
from risk_management.measurements import compute_correlation, compute_covariance, compute_returns
from risk_management.goodness_of_fit import log_likelihood

In [8]:
returns = compute_returns(data['Close'])

In [9]:
corr = compute_correlation(returns)

In [10]:
corr

Ticker,DBA,DBC,FXE,FXY,GLD,UNG,USO,UUP,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Ticker,,,,,,,,,,,,,,,,,,,
DBA,1.000000,0.446488,0.123941,0.033445,0.175158,0.093902,0.289610,-0.148966,0.214947,0.128224,0.258830,0.159766,0.174615,0.141578,0.075373,0.110167,0.054879,0.063100,0.120617
DBC,0.446488,1.000000,0.135353,0.031474,0.338342,0.248376,0.919956,-0.184628,0.306626,0.152423,0.658966,0.215338,0.260127,0.183917,0.083881,0.135274,0.123550,0.080194,0.157161
FXE,0.123941,0.135353,1.000000,0.481657,0.414726,-0.001358,0.039723,-0.957535,0.336415,0.209672,0.103003,0.213290,0.223460,0.198951,0.230739,0.277981,0.170067,0.214029,0.223511
FXY,0.033445,0.031474,0.481657,1.000000,0.394661,-0.026032,-0.040131,-0.629276,0.057531,0.004424,-0.058370,-0.086771,-0.044254,-0.048293,0.094881,0.141682,0.125457,0.054295,-0.022056
GLD,0.175158,0.338342,0.414726,0.394661,1.000000,0.030950,0.193884,-0.474101,0.228391,0.099989,0.157851,0.029379,0.110143,0.112967,0.122030,0.191078,0.226096,0.107790,0.064076
UNG,0.093902,0.248376,-0.001358,-0.026032,0.030950,1.000000,0.130612,-0.016773,0.120766,0.071277,0.187227,0.100100,0.091959,0.084787,0.057966,0.090175,0.085491,0.082941,0.082526
USO,0.289610,0.919956,0.039723,-0.040131,0.193884,0.130612,1.000000,-0.073732,0.225619,0.107065,0.656849,0.186242,0.212370,0.126925,0.037205,0.080929,0.071756,0.037041,0.103066
UUP,-0.148966,-0.184628,-0.957535,-0.629276,-0.474101,-0.016773,-0.073732,1.000000,-0.373373,-0.238278,-0.139347,-0.225479,-0.247694,-0.218744,-0.262152,-0.322262,-0.217418,-0.241660,-0.248575
XLB,0.214947,0.306626,0.336415,0.057531,0.228391,0.120766,0.225619,-0.373373,1.000000,0.615750,0.548355,0.791578,0.855675,0.627099,0.563214,0.650546,0.482160,0.622953,0.677872


In [11]:
cov = compute_covariance(returns)

In [12]:
cov

Ticker,DBA,DBC,FXE,FXY,GLD,UNG,USO,UUP,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Ticker,,,,,,,,,,,,,,,,,,,
DBA,0.000083,0.000048,5.442184e-06,1.921879e-06,0.000016,3.291212e-05,0.000055,-0.000006,0.000023,1.540513e-05,0.000040,0.000017,0.000017,0.000020,0.000006,0.000012,0.000005,0.000005,0.000016
DBC,0.000048,0.000140,7.722215e-06,2.349960e-06,0.000039,1.131106e-04,0.000227,-0.000010,0.000043,2.379366e-05,0.000132,0.000030,0.000033,0.000034,0.000008,0.000019,0.000016,0.000009,0.000028
FXE,0.000005,0.000008,2.332955e-05,1.470557e-05,0.000020,-2.527955e-07,0.000004,-0.000021,0.000019,1.338395e-05,0.000008,0.000012,0.000012,0.000015,0.000009,0.000016,0.000009,0.000009,0.000016
FXY,0.000002,0.000002,1.470557e-05,3.995593e-05,0.000024,-6.344154e-06,-0.000005,-0.000018,0.000004,3.695404e-07,-0.000006,-0.000007,-0.000003,-0.000005,0.000005,0.000011,0.000009,0.000003,-0.000002
GLD,0.000016,0.000039,1.958385e-05,2.438924e-05,0.000096,1.166588e-05,0.000040,-0.000021,0.000027,1.291897e-05,0.000026,0.000003,0.000012,0.000017,0.000010,0.000023,0.000024,0.000010,0.000009
UNG,0.000033,0.000113,-2.527955e-07,-6.344154e-06,0.000012,1.486433e-03,0.000105,-0.000003,0.000056,3.631742e-05,0.000122,0.000046,0.000038,0.000051,0.000018,0.000042,0.000036,0.000029,0.000048
USO,0.000055,0.000227,4.004088e-06,-5.293965e-06,0.000040,1.050908e-04,0.000436,-0.000007,0.000056,2.952882e-05,0.000233,0.000046,0.000048,0.000041,0.000006,0.000020,0.000016,0.000007,0.000032
UUP,-0.000006,-0.000010,-2.095929e-05,-1.802604e-05,-0.000021,-2.930648e-06,-0.000007,0.000021,-0.000020,-1.427065e-05,-0.000011,-0.000012,-0.000012,-0.000015,-0.000010,-0.000018,-0.000011,-0.000010,-0.000017
XLB,0.000023,0.000043,1.941415e-05,4.344910e-06,0.000027,5.562981e-05,0.000056,-0.000020,0.000143,9.722675e-05,0.000111,0.000113,0.000111,0.000116,0.000055,0.000094,0.000063,0.000068,0.000122


In [13]:
U = pd.DataFrame([{"A": 0.2, "B":0.3}, {"A": 0.4, "B":0.6}, {"A": 0.4, "B":0.9}, {"A": 0.99, "B":0.1}])
LLs = []
for v in [2, 3, 4, 5, 7, 9, 12, 16, 20, 25, 30, 36, 42, 50]:
    T = t.ppf(U, df=v)
    print("DF:", v)
    corr = compute_correlation(pd.DataFrame(T), method="pearson")
    print("CORR:", corr)
    t_params = {"loc": np.zeros(corr.shape[0]), "shape": corr, "df":v}
    LL = log_likelihood(multivariate_t.pdf, t_params, T)
    print("LL:", LL)
    LLs.append((LL, v))

DF: 2
CORR:           0         1
0  1.000000 -0.705583
1 -0.705583  1.000000
LL: -18.303870124825337
DF: 3
CORR:          0        1
0  1.00000 -0.67654
1 -0.67654  1.00000
LL: -15.889956769112231
DF: 4
CORR:           0         1
0  1.000000 -0.660923
1 -0.660923  1.000000
LL: -14.70281006598284
DF: 5
CORR:           0         1
0  1.000000 -0.651443
1 -0.651443  1.000000
LL: -14.007302830130751
DF: 7
CORR:           0         1
0  1.000000 -0.640672
1 -0.640672  1.000000
LL: -13.234756120809113
DF: 9
CORR:          0        1
0  1.00000 -0.63477
1 -0.63477  1.00000
LL: -12.818038008599355
DF: 12
CORR:          0        1
0  1.00000 -0.62968
1 -0.62968  1.00000
LL: -12.461651856986963
DF: 16
CORR:           0         1
0  1.000000 -0.625917
1 -0.625917  1.000000
LL: -12.199730376598083
DF: 20
CORR:           0         1
0  1.000000 -0.623685
1 -0.623685  1.000000
LL: -12.04486718556257
DF: 25
CORR:           0         1
0  1.000000 -0.621913
1 -0.621913  1.000000
LL: -11.922238042353

In [14]:
max(LLs)

(np.float64(-11.68039391953555), 50)

### Playing with the Gaussian Copula

In [15]:
from copulae.elliptical import GaussianCopula as copulae_g
from statsmodels.distributions.copula.api import GaussianCopula as sm_g

In [16]:
g_copula = copulae_g(dim=19)
g_copula.fit(U, to_pobs=False, method="ml", verbose=1000)

InputDataError: Dimension of data does not match copula

In [ ]:
rho_mat = g_copula.params
g_corr = construct_corr_from_rhos(dim=19, rho_array=rho_mat)

In [ ]:
g_copula.random(1)

NameError: name 'g_copula' is not defined

In [ ]:
# g_copula.sigma -> same as g_corr
g_copula.cdf(np.random.rand(19))

8.675120289083417e-29

In [ ]:
mask = ~np.eye(g_corr.shape[0], dtype=bool)
off_diag_indices = np.argwhere(mask)
max_idx = g_corr[mask].argmax()
i, j = off_diag_indices[max_idx]
g_corr[i, j]


np.float64(0.9141865699893618)

In [ ]:
g_corr

array([[ 1.        ,  0.44542248,  0.1128978 ,  0.0216103 ,  0.16510532,
         0.09167549,  0.28025041, -0.13930141,  0.20928074,  0.1150262 ,
         0.24443808,  0.15496742,  0.16034499,  0.12635351,  0.05906388,
         0.09772575,  0.04624597,  0.05295227,  0.11257122],
       [ 0.44542248,  1.        ,  0.11733405,  0.00673382,  0.3242407 ,
         0.24775971,  0.91418657, -0.16721317,  0.29883015,  0.14595918,
         0.6620589 ,  0.21044169,  0.24087309,  0.16890944,  0.05906618,
         0.10769307,  0.0975625 ,  0.05627586,  0.14651778],
       [ 0.1128978 ,  0.11733405,  1.        ,  0.41268768,  0.39143534,
        -0.02999757,  0.02571832, -0.95115659,  0.31746512,  0.21073621,
         0.09363513,  0.22218505,  0.22873439,  0.1997484 ,  0.207295  ,
         0.24654639,  0.12855506,  0.21964918,  0.22000047],
       [ 0.0216103 ,  0.00673382,  0.41268768,  1.        ,  0.39473384,
        -0.04263989, -0.06266785, -0.57059573,  0.01884235, -0.02104834,
        -0.084

### Making the T copula

In [ ]:
# before this, compute the U for each asset based on its assumed distribution

def fit_student_t_copula(U: pd.DataFrame):

    # apply inverse CDF of univariate T to each entry (column) in U
    # T = tv ^-1 (U) -> do this to each column
    T = t.ppf(U, df=2)
    # TODO -> do the T thing for many different values of v, then compute correlation and compute log likelihood. Choose one with best (highest) log likelihood
        # Correlation is done how we would usually do it
    
    # TODO -> choose a guess of v from [2, 3, 4, 5, 7, 9, 12, 16, 20, 25, 30, 36, 42, 50]
        # then compute correlation and the log likelihood
    
    # then, optimize further using scipy.optimize.minimize_scalar

    
    # return corr, df
    pass

def simulate_from_t_copula(corr, df, n_sims):
    # in order to simulate from a t copula, you take the multivariate T, simulate values, and convert them to uniform using the standard t (with fitted df)
    pass

### Using Package Copulas

In [17]:
from copulae.elliptical import StudentCopula as copulae_t
from statsmodels.distributions.copula.api import StudentTCopula as sm_t

In [18]:
# fit normal distributions
returns_np = returns.to_numpy()
means = returns_np.mean(axis=0)
stds = returns_np.std(axis=0)



In [19]:
returns_np[:, 0]

array([-0.00520157, -0.00065361, -0.00327013, ...,  0.01109839,
       -0.00492048,  0.00456443], shape=(1254,))

In [20]:
t_params_by_col = {}
Us = []
for i in range(returns_np.shape[1]):
    nu, mu, sigma = t.fit(returns_np[:, i])
    U_col = t.cdf(returns_np[:, i], loc=mu, scale=sigma, df=nu)
    Us.append(U_col)
    t_params_by_col[i] = (mu, sigma, nu)

U = np.array(Us).T

In [21]:
# U = norm.cdf(returns_np - means / stds)
# t.cdf()

In [22]:
sm_t_copula = sm_t(df=2, k_dim=19)
sm_t_copula.corr = sm_t_copula.fit_corr_param(data=U)

In [23]:
sm_t_copula.rvs(1)

array([0.65208697, 0.65042254, 0.20802524, 0.24392893, 0.73880628,
       0.64242703, 0.4331701 , 0.78384546, 0.86491794, 0.69940331,
       0.42920258, 0.35932599, 0.92225927, 0.6082398 , 0.50777844,
       0.05404165, 0.1998259 , 0.58171084, 0.12668686])

In [24]:
corrs = {}
LLs = {}
for v in range(1, 50):
    sm_t_copula = sm_t(df=v, k_dim=19)
    sm_t_copula.corr = sm_t_copula.fit_corr_param(data=U)
    LL = np.sum(sm_t_copula.logpdf(U))
    corrs[v] = sm_t_copula.corr
    LLs[v] = LL
    # print(f"LL {v}", LL)

# maximum likelihood v
df, ll = max(LLs.items(), key=lambda x: x[1])
sm_t_copula = sm_t(df=df, k_dim=19)
sm_t_copula.corr = sm_t_copula.fit_corr_param(data=U)

In [25]:
sm_t_copula.df

5

In [26]:
t_copula = copulae_t(dim=19)
t_copula.fit(U, to_pobs=False, method="ml", verbose=1000)

Optimization terminated successfully    (Exit mode 0)
            Current function value: -10394.214446174912
            Iterations: 182
            Function evaluations: 32065
            Gradient evaluations: 182


In [27]:
sm_t_copula.df - t_copula.params.df

np.float64(-7.9481275038478465)

In [ ]:
t_copula.params.df

np.float64(14.010875470701492)

In [ ]:
sm_t_copula.df

5

In [ ]:
t_copula.random(100000)

(100000, 19)

In [ ]:
t_copula.rho.shape # 19x18/2 (diagonals are assumed 1 and thus not included)
t_copula.dim

19

In [ ]:
t_copula

In [ ]:
t_copula.summary()

1.000000,0.422842,0.122781,0.036579,0.150698,0.079008,0.253870,-0.146528,0.201148,0.113494,0.222259,0.150784,0.150478,0.123797,0.067063,0.098836,0.050393,0.053664,0.111655
0.422842,1.000000,0.142479,0.015535,0.312558,0.244392,0.912043,-0.189704,0.296145,0.150364,0.653387,0.214716,0.235722,0.164848,0.061786,0.102655,0.083114,0.055442,0.146202
0.122781,0.142479,1.000000,0.420548,0.401445,-0.027896,0.044135,-0.950558,0.342464,0.241617,0.121017,0.233853,0.256301,0.227946,0.220292,0.264243,0.145926,0.240524,0.247260
0.036579,0.015535,0.420548,1.000000,0.400670,-0.038389,-0.057802,-0.576951,0.045860,0.004015,-0.067814,-0.093839,-0.043016,-0.050539,0.086054,0.130828,0.110113,0.058700,-0.023679
0.150698,0.312558,0.401445,0.400670,1.000000,0.022970,0.163111,-0.469172,0.221668,0.101968,0.144988,0.029274,0.102347,0.099505,0.104707,0.172102,0.202012,0.103353,0.056401
0.079008,0.244392,-0.027896,-0.038389,0.022970,1.000000,0.126239,0.008692,0.098769,0.051252,0.173267,0.083259,0.071070,0.067973,0.050770,0.069118,0.062772,0.058482,0.072015
0.253870,0.912043,0.044135,-0.057802,0.163111,0.126239,1.000000,-0.075043,0.202171,0.094909,0.645613,0.169930,0.176191,0.093355,0.011401,0.046604,0.033456,0.009933,0.081015
-0.146528,-0.189704,-0.950558,-0.576951,-0.469172,0.008692,-0.075043,1.000000,-0.383473,-0.268900,-0.149604,-0.243907,-0.277528,-0.245370,-0.250383,-0.309627,-0.189473,-0.261411,-0.272398
0.201148,0.296145,0.342464,0.045860,0.221668,0.098769,0.202171,-0.383473,1.000000,0.595099,0.523570,0.775138,0.844041,0.592741,0.552942,0.625283,0.458373,0.600901,0.653749
0.113494,0.150364,0.241617,0.004015,0.101968,0.051252,0.094909,-0.268900,0.595099,1.000000,0.264368,0.615587,0.639948,0.769849,0.414546,0.511508,0.327318,0.496512,0.760797
0.222259,0.653387,0.121017,-0.067814,0.144988,0.173267,0.645613,-0.149604,0.523570,0.264368,1.000000,0.531760,0.505564,0.239519,0.228281,0.268712,0.216059,0.248324,0.274344


In [ ]:
t_copula.sigma

array([[ 1.        ,  0.42284214,  0.12278061,  0.03657923,  0.15069756,
         0.07900843,  0.25386976, -0.14652808,  0.20114831,  0.11349402,
         0.22225908,  0.1507839 ,  0.15047803,  0.12379667,  0.06706254,
         0.09883634,  0.05039278,  0.05366405,  0.11165535],
       [ 0.42284214,  1.        ,  0.14247913,  0.01553547,  0.3125583 ,
         0.2443924 ,  0.9120426 , -0.18970402,  0.29614525,  0.15036354,
         0.65338689,  0.21471595,  0.23572219,  0.16484846,  0.06178594,
         0.10265468,  0.08311378,  0.05544207,  0.14620223],
       [ 0.12278061,  0.14247913,  1.        ,  0.42054818,  0.40144542,
        -0.02789553,  0.04413494, -0.95055844,  0.34246353,  0.2416166 ,
         0.12101651,  0.23385252,  0.25630058,  0.22794578,  0.22029161,
         0.26424259,  0.1459262 ,  0.24052394,  0.24726009],
       [ 0.03657923,  0.01553547,  0.42054818,  1.        ,  0.40066984,
        -0.03838877, -0.05780186, -0.57695079,  0.04585987,  0.00401497,
        -0.067

In [ ]:
def construct_corr_from_rhos(dim, rho_array):
    upper_triangular = np.zeros((dim, dim))
    indices = np.triu_indices(dim, k=1)
    upper_triangular[indices] = rho_array
    corr = upper_triangular + upper_triangular.T + np.eye(dim)

    return corr

In [ ]:
def compute_tail_dependence(df, corr):

    def t_tail_dependence(df, rho):
        # df = degrees of freedom
        # rho = correlation between variables i,j
        arg = -np.sqrt((df + 1) * (1 - rho) / (1 + rho))
        return 2 * t.cdf(arg, df + 1)

    # Example for all pairs in a fitted copula:
    df = t_copula.params.df

    tail_dep_matrix = np.zeros_like(corr)

    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            tail_dep_matrix[i, j] = t_tail_dependence(df, corr[i, j])
        
    return tail_dep_matrix

In [ ]:
from numpy.linalg import eigh

# eigh(full_rho)
# eigh(t_copula.sigma)

EighResult(eigenvalues=array([0.02740576, 0.05025982, 0.11780792, 0.17946242, 0.19534363,
       0.22243164, 0.26469093, 0.29604041, 0.36413637, 0.45536438,
       0.52309736, 0.56533173, 0.75474515, 0.85358383, 0.94671051,
       1.28939898, 2.38945408, 2.53833914, 6.96639593]), eigenvectors=array([[ 6.79416655e-03, -1.24923311e-01,  3.04353742e-02,
        -1.22112111e-03,  2.33550487e-02,  5.70454472e-02,
         4.64010183e-02,  6.67068894e-02,  2.06408871e-02,
        -7.19455295e-03, -6.60872872e-02,  2.78120598e-02,
        -2.87316894e-01,  8.93161019e-01, -3.21216939e-02,
         5.90155000e-02, -5.96044432e-02,  2.66445151e-01,
         8.97268366e-02],
       [-7.53792808e-02,  7.35374547e-01, -3.59407307e-02,
        -8.81467354e-03,  6.12924083e-03, -1.59330928e-01,
        -1.39324614e-01, -9.70478364e-02, -3.27979550e-03,
        -2.63117857e-02,  1.26022687e-01, -1.78752537e-01,
         1.33215536e-01,  6.90762209e-03,  9.53516972e-03,
        -3.43272690e-02, -2.028

In [ ]:
full_rho

array([[ 1.        ,  0.40685479,  0.11732062,  0.03493254,  0.14404208,
         0.07546713,  0.24308355, -0.14004949,  0.1924078 ,  0.10843706,
         0.21268109,  0.14412477,  0.14383185,  0.11829273,  0.06405202,
         0.09442021,  0.04812666,  0.05125154,  0.10667847],
       [ 0.40685479,  1.        ,  0.13617289,  0.01483543,  0.29969967,
         0.23396228,  0.90435552, -0.18142674,  0.28384163,  0.14372221,
         0.63560678,  0.20543456,  0.22562254,  0.15759748,  0.05901062,
         0.09807109,  0.07939068,  0.05295006,  0.13973749],
       [ 0.11732062,  0.13617289,  1.        ,  0.40461384,  0.38597415,
        -0.02663913,  0.04214918, -0.94591772,  0.32864814,  0.23129181,
         0.11563289,  0.22382471,  0.24542392,  0.21814622,  0.21079069,
         0.25307303,  0.1394732 ,  0.23024073,  0.23672165],
       [ 0.03493254,  0.01483543,  0.40461384,  1.        ,  0.38521817,
        -0.03666083, -0.0552044 , -0.55889007,  0.04379679,  0.00383402,
        -0.064

In [ ]:
full_rho - t_copula.sigma

array([[ 0.        , -0.0159325 , -0.00588002, -0.00240428, -0.00700338,
        -0.00353599, -0.010654  ,  0.00695556, -0.00887771, -0.00529628,
        -0.00957799, -0.00673864, -0.00672082, -0.00564983, -0.00316471,
        -0.00467962, -0.00238505, -0.00256503, -0.0052391 ],
       [-0.0159325 ,  0.        , -0.007576  , -0.0022197 , -0.01319552,
        -0.01042976, -0.00778305,  0.00947084, -0.01260997, -0.00701867,
        -0.01774921, -0.00953525, -0.01031783, -0.00741732, -0.00307457,
        -0.00499898, -0.00386798, -0.00275155, -0.00687421],
       [-0.00588002, -0.007576  ,  0.        , -0.01727591, -0.01616944,
         0.00038706, -0.00308066,  0.00391916, -0.0145576 , -0.01112211,
        -0.00618896, -0.01050634, -0.01149014, -0.01030997, -0.01034889,
        -0.01223419, -0.00787768, -0.01076248, -0.01136076],
       [-0.00240428, -0.0022197 , -0.01727591,  0.        , -0.01598124,
         0.00121913,  0.00145228,  0.017966  , -0.00407506, -0.00181661,
         0.001

In [ ]:
# t_copula.summary()

In [ ]:
dir(t_copula)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_bounds',
 '_columns',
 '_df',
 '_dim',
 '_fit_smry',
 '_force_psd',
 '_name',
 '_rhos',
 'bounds',
 'cdf',
 'dim',
 'drho',
 'dtau',
 'fit',
 'irho',
 'itau',
 'lambda_',
 'log_lik',
 'name',
 'params',
 'pdf',
 'pobs',
 'random',
 'rho',
 'sigma',
 'summary',
 'tau']

In [ ]:
pd.DataFrame(t_copula.sigma - sm_t_copula.corr)

np.float64(-0.03147638389496016)

In [ ]:
np.equal(corrs[1], corrs[20]).all()

np.True_

In [ ]:
low_tail, high_tail = sm_t_copula.dependence_tail()

In [ ]:
sm_t_copula = sm_t(df=50, k_dim=19)
sm_t_copula.corr = sm_t_copula.fit_corr_param(data=U)
sm_t_copula.corr[0]

array([ 1.        ,  0.32637347,  0.12185952,  0.03052481,  0.13780138,
        0.07492821,  0.18344942, -0.13467134,  0.26583224,  0.25870027,
        0.2221099 ,  0.12474359,  0.26596678,  0.22760942,  0.01419645,
        0.13811707,  0.12357637,  0.06147527,  0.30923474])

In [ ]:
sm_t_copula.df

10

In [ ]:
# sm_t_copula.corr - sm_t_copula.spearmans_rho()

In [ ]:
# sm_t_copula.spearmans_rho()

In [ ]:
# sm_t_copula.corr

In [ ]:
# sm_t_copula.corr

In [ ]:
sm_t_copula

array([[1.        , 0.32637347],
       [0.32637347, 1.        ]])

In [ ]:
t_copula = copulae_t()

In [ ]:
returns.to_numpy().shape

(249, 19)

In [ ]:
t_copula = copulae_t(dim=19)
t_copula.fit(returns.to_numpy(), to_pobs=True, method="ml")

In [ ]:
t_copula.params.df, t_copula.sigma.shape

(np.float64(9.726471626704669), (19, 19))

In [ ]:
sm_t_copula.df - t_copula.params.df

np.float64(39.273528373295335)

In [ ]:
t_copula.sigma - sm_t_copula.corr
sm_t_copula.df - t_copula.params.df

array([[ 0.        , -0.1274381 , -0.01511036, -0.01652867, -0.01344375,
        -0.00866895, -0.08522274,  0.03551505,  0.05134306,  0.11775149,
        -0.00279799, -0.04194926,  0.09722427,  0.1022937 , -0.03647307,
         0.06658016,  0.08775792,  0.01548735,  0.1599504 ],
       [-0.1274381 ,  0.        , -0.21730775, -0.1539832 , -0.02455612,
         0.04349179, -0.03265089,  0.24940647, -0.15742196, -0.06436268,
        -0.03265139, -0.18904887, -0.12982943,  0.01204029, -0.12349707,
        -0.02438365,  0.01779207, -0.10370022, -0.03842564],
       [-0.01511036, -0.21730775,  0.        ,  0.09985651, -0.00523925,
        -0.04458467, -0.24793068, -0.00724592, -0.16462728, -0.24098417,
        -0.17552749, -0.2466614 , -0.22053975, -0.26373857, -0.08556595,
        -0.06706415, -0.04761868, -0.1170964 , -0.25834565],
       [-0.01652867, -0.1539832 ,  0.09985651,  0.        , -0.08168139,
        -0.00076988, -0.16368871, -0.08362323, -0.04130768, -0.14987019,
        -0.040

In [ ]:
for column in returns:
    data = returns[column]
    